In [ ]:
from util.load_battery_config import BatteryConfigJupyter


# Usage example
if __name__ == "__main__":
    # Create and display the configuration UI
    config_ui = BatteryConfigJupyter()
    config_ui.display()

In [ ]:

from minio import Minio
from minio.error import S3Error
import urllib3
import io
import pandas as pd
import os

minio_endpoint = "iseadocker.isea.rwth-aachen.de:9000"
access_key= "21HzJu0TMxt0wwqDdzli"
secret_key= "BRR0TWeZCLBh9SlWMLjdpMrxdDQvYLO8okBFqXKp"
bucket_name= "projects"
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def export_to_server(self, df, object_name):

    parquet_buffer = io.BytesIO()
    df.to_parquet(parquet_buffer, index=False)
    parquet_buffer.seek(0)

    try:
        self.minio_client.put_object(
            bucket_name=self.bucket_name,
            object_name=object_name,
            data=parquet_buffer,
            length=len(parquet_buffer.getvalue()),
        )
        print("File uploaded successfully.")

    except S3Error as err:
        print("Upload error:", err)

if access_key and secret_key and minio_endpoint:
    minio_client = Minio(
        minio_endpoint,
        access_key=access_key,
        secret_key=secret_key,
        secure=True,
        cert_check=False,
    )

objects = minio_client.list_objects(
    bucket_name=bucket_name,
    prefix=f"j8005-metabatt/Metabatt/{type_cell}/",
    recursive=True,
)

folders = set()
prefix = f"j8005-metabatt/Metabatt/{type_cell}/"

for obj in objects:
    # Remove the prefix
    path_after_prefix = obj.object_name[len(prefix):]
    
    # Get the first folder name
    if '/' in path_after_prefix:
        folder_name = path_after_prefix.split('/')[0]
        folders.add(folder_name)

cells = sorted(list(folders))

In [ ]:
cells

# MinIO

In [ ]:

export_type = "local"

for cell in cells:
    savepath_df_BRONZE = os.path.join(
        working_path, "BRONZE_CU", cell + ".parquet"
    )
    if not os.path.exists(savepath_df_BRONZE):
        objects = minio_client.list_objects(
            bucket_name=bucket_name,
            prefix=f"j8005-metabatt/Metabatt/{type_cell}/{cell}/",
            recursive=True,
        )
        
        # Get list of parquet files
        cell_tests = [obj.object_name for obj in objects if obj.object_name.endswith('.parquet')]
        
        if any("jri_CU" in test for test in cell_tests):

            tests = []
            try:
                for test_file in cell_tests:
                    programme_name = os.path.basename(test_file).split('=')[3]
                    
                    # Get file from MinIO
                    response = minio_client.get_object(bucket_name, test_file)
                    data = response.read()
                    response.close()
                    response.release_conn()
                    
                    # Load only first row quickly
                    df_first_row = pd.read_parquet(io.BytesIO(data)).head(1)
                    
                    # Load whole parquet file if test is a Checkup
                    if "jri_CU" in programme_name:
                        df = pd.read_parquet(io.BytesIO(data))
                    # Load only first row otherwise
                    else:
                        df = df_first_row
                        df["Prozedur"] = programme_name  # Add procedure name to column
                    
                    tests.append(df)
            
            except Exception as e:
                print(f"Error loading {test_file}: {e}. Skipping file.")
                continue
            
            if tests:  # Only save if we have data
                cell_df = pd.concat(tests, ignore_index=True)
                if export_type == "server":
                    object_name = f"Metabatt/BRONZE/{cell}_CU_only.parquet"
                    export_to_server(minio_client, cell_df, object_name)
                    
                elif export_type == "local":
                    save_path = os.path.join(rootpath+'_CU', cell+'.parquet')
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    cell_df.to_parquet(save_path)


# Local

In [ ]:
import io
import os
import pandas as pd
import duckdb
import glob

for cell in cells:
    loadpath = os.path.join(rootpath, cell)
    cell_tests = glob.glob('*.parquet', root_dir=loadpath)

    if any ("jri_CU" in test for test in cell_tests):
        tests = []
        try:
            for test_file in cell_tests:
                programme_name = os.path.basename(test_file).split('=')[3]
                filepath = os.path.join(loadpath, test_file)
                # Load only first row quickly using duckdb
                df_first_row = duckdb.sql(f"SELECT * FROM '{filepath}' LIMIT 1").df()

                # Load whole parquet file if test is a Checkup
                if "jri_CU" in programme_name:
                    df = pd.read_parquet(filepath)
                # Load only first row otherwise
                else:
                    df = df_first_row

                tests.append(df)

        except Exception as e:
            print(f"Error loading {filepath}: {e}. Removing file.")
            os.remove(filepath)
            continue
        
        cell_df = pd.concat(tests, ignore_index=True)
        save_path = os.path.join(rootpath+'_CU', cell+'.parquet')
        cell_df.to_parquet(save_path)
